# 🚨 Notebook 13: Early Warning System (Extension 3)

**Objective:** Implement and evaluate the threshold-based early-warning alert module.

- Alert when predicted AQI is **Very Poor** (code 4) or **Severe** (code 5)
- Evaluate alert precision, recall, and F1-score
- Generate sample health advisory messages

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config
from src.preprocessing.sequence_generator import load_sequences
from src.prediction.early_warning import (
    generate_alerts, evaluate_early_warning, format_alert_message
)
from src.prediction.health_risk import (
    add_health_risk_columns, get_risk_summary, HEALTH_RISK_MAP
)

config = load_config('../config/config.yaml')
X_train, X_test, y_train, y_test = load_sequences(
    '../' + config['data']['sequences_path']
)

# Use predictions from the best model (Hybrid CNN-LSTM)
y_pred = np.load('../artifacts/y_pred_hybrid.npy')
print(f"Test samples: {len(y_test)}")
print(f"Predictions: {len(y_pred)}")

## 13.1 Generate Alerts

In [ ]:
# Generate binary alerts from predictions
pred_alerts = generate_alerts(y_pred)
true_alerts = generate_alerts(y_test)

print(f"\nTrue high-risk episodes: {true_alerts.sum()}")
print(f"Predicted high-risk episodes: {pred_alerts.sum()}")

## 13.2 Evaluate Alert System

In [ ]:
# Evaluate early warning reliability
alert_metrics = evaluate_early_warning(y_test, y_pred)

In [ ]:
# Visualize alert confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Alert confusion pie chart
labels = ['True Positive\n(Correct Alert)', 'False Positive\n(False Alarm)',
          'False Negative\n(Missed Alert)', 'True Negative\n(Correct Clear)']
sizes = [alert_metrics['true_positives'], alert_metrics['false_positives'],
         alert_metrics['false_negatives'], alert_metrics['true_negatives']]
colors_pie = ['#4CAF50', '#FF9800', '#F44336', '#2196F3']
axes[0].pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 9})
axes[0].set_title('Early Warning — Alert Distribution')

# Metrics bar chart
metric_names = ['Precision', 'Recall', 'F1-Score']
metric_values = [alert_metrics['alert_precision'],
                 alert_metrics['alert_recall'],
                 alert_metrics['alert_f1_score']]
bars = axes[1].bar(metric_names, metric_values, color=['#2196F3', '#FF9800', '#4CAF50'])
for bar, val in zip(bars, metric_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Score')
axes[1].set_title('Early Warning System — Performance Metrics')
axes[1].grid(True, alpha=0.2, axis='y')

plt.suptitle('Early Warning Alert System Evaluation', fontsize=14)
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/early_warning_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 13.3 Health Risk Mapping

In [ ]:
# Create a DataFrame with predictions and health risk info
pred_df = pd.DataFrame({
    'true_aqi_code': y_test,
    'predicted_aqi_code': y_pred,
})

pred_df = add_health_risk_columns(pred_df, aqi_code_col='predicted_aqi_code')

# Risk summary
risk_summary = get_risk_summary(pred_df, aqi_code_col='predicted_aqi_code')
print("\nHealth Risk Distribution:")
print(risk_summary.to_string(index=False))

## 13.4 Sample Alert Messages

In [ ]:
# Generate sample alerts for high-risk predictions
alert_indices = np.where(pred_alerts == 1)[0][:5]  # First 5 alerts

print("\n" + "=" * 70)
print("  SAMPLE EARLY WARNING ALERTS")
print("=" * 70)

for idx in alert_indices:
    alert = format_alert_message(y_pred[idx])
    if alert:
        print(f"\n  [{alert['level']}] {alert['message']}")
        print(f"  Category: {alert['category']}")
        print(f"  Advisory: {alert['health_advisory']}")
        print(f"  {'─' * 60}")

print("\n" + "=" * 70)

## 13.5 Save Alert Results

In [ ]:
# Save alert metrics
import json
with open('../results/early_warning_metrics.json', 'w') as f:
    json.dump(alert_metrics, f, indent=2)

# Save prediction DataFrame
pred_df.to_csv('../results/predictions_with_health_risk.csv', index=False)

print("Early warning results saved!")
print(f"  Metrics: results/early_warning_metrics.json")
print(f"  Predictions: results/predictions_with_health_risk.csv")

## ✅ Early Warning System Complete

**All 13 notebooks complete!** The project pipeline is fully implemented:
1. ✅ Data Collection
2. ✅ Data Preprocessing
3. ✅ AQI Calculation
4. ✅ Feature Engineering
5. ✅ Exploratory Data Analysis
6. ✅ Sequence Creation
7. ✅ CNN Model
8. ✅ LSTM Model
9. ✅ Hybrid CNN-LSTM Model
10. ✅ Ablation Study (Extension 2)
11. ✅ Meteorological Extension (Extension 1)
12. ✅ Model Evaluation
13. ✅ Early Warning System (Extension 3)